# INSTRUCTOR SOLUTION: FastAPI Deployment
## AIAT 125

**INSTRUCTOR ONLY** — minimal runnable API + TestClient.


In [ ]:
%pip install fastapi uvicorn pydantic numpy -q
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

class SentimentModel:
    def predict(self, text: str) -> float:
        return float(len(text) % 2)

model = SentimentModel()
app = FastAPI(title='Sentiment API', version='1.0.0')

class TextIn(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)

class ScoreOut(BaseModel):
    score: float
    label: str

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/predict', response_model=ScoreOut)
def predict(body: TextIn):
    try:
        s = model.predict(body.text)
        return ScoreOut(score=s, label='pos' if s >= 0.5 else 'neg')
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

c = TestClient(app)
assert c.get('/health').status_code == 200
print(c.post('/predict', json={'text': 'hello'}).json())
print('OK')
